## Classic Environment Preflight

This notebook requires the classic runtime. If this check fails, rebuild with `CLASSIC=1 make notebooks-build`, restart the container, and select kernel **Python 3 (classic-langchain)**.


In [ ]:
import os
import sys

def _classic_fail(reason: str) -> None:
    raise RuntimeError(
        f"Classic runtime preflight failed: {reason}\n"
        "Fix:\n"
        "1. CLASSIC=1 make notebooks-build\n"
        "2. make notebooks-up\n"
        "3. In Jupyter, select kernel: Python 3 (classic-langchain)"
    )

kernel_name = os.getenv("JPY_KERNEL_NAME", "")
prefix = sys.prefix.lower()
if "venv-classic" not in prefix and "classic" not in kernel_name.lower():
    _classic_fail(f"detected sys.prefix={sys.prefix!r}, JPY_KERNEL_NAME={kernel_name!r}")

try:
    import langchain  # noqa: F401
except Exception as exc:
    _classic_fail(f"langchain import failed: {exc}")

print("Classic preflight passed.")


To reduce the costs we can consider caching the results of the llm.
Langchain allows us to do global caching of calls.
You need to verify that the caching does not contain personalized answers that should not be cached.
> Classic track note: This notebook demonstrates legacy/classic LangChain-era patterns for evaluation and comparison.
> Prefer the modern equivalents in `lessons/2026/langchain/` for current APIs and recommended techniques.


In [ ]:
%pip install -q langchain langchain-openai gptcache
%pip install -q --no-cache-dir "onnxruntime>=1.14.1"

In [ ]:
%pip install -q python-dotenv
from dotenv import load_dotenv
load_dotenv()

In the first strategy we enable the caching on a global level.
We use a database to store the input prompt and output prompt.
And when we get another request that is the same , we return it from cache.

In [ ]:
from gptcache import Cache
from gptcache.manager.factory import manager_factory
from gptcache.processor.pre import get_prompt

from langchain.cache import GPTCache
import langchain
import hashlib


def get_hashed_name(name):
    return hashlib.sha256(name.encode()).hexdigest()


def init_gptcache_exact_match(cache_obj: Cache, llm: str):
    hashed_llm = get_hashed_name(llm)
    cache_obj.init(
        pre_embedding_func=get_prompt,
        data_manager=manager_factory(manager="map", data_dir=f"map_cache_{hashed_llm}"),
    )


def _normalize_prompt(data, *args, **kwargs):
    prompt = get_prompt(data)
    return " ".join(prompt.lower().replace(":)", "").split())


def init_gptcache_embeddings_match(cache_obj: Cache, llm: str):
    # Lightweight semantic-ish cache keying that avoids heavy embedding runtimes.
    hashed_llm = get_hashed_name(llm + "_normalized")
    cache_obj.init(
        pre_embedding_func=_normalize_prompt,
        data_manager=manager_factory(manager="map", data_dir=f"similar_cache_{hashed_llm}"),
    )


Running the same prompt 10 times is slow

In [ ]:

from langchain_openai import OpenAI
llm=OpenAI(temperature=0)
prompt="Hello world"

langchain.llm_cache=None
for i in range(1,10):
    result = llm.invoke(prompt)
   # print(result)


When we enable the caching it goes a lot faster once it's warmed up

In [ ]:
langchain.llm_cache = GPTCache(init_gptcache_exact_match)
# Now run it once
result = llm.invoke("Hello world")


In [ ]:

# Now run it 10 times
for i in range(1,10):
    result = llm.invoke(prompt)
    #print(result)

With embeddings we can make this is a bit more clever. Not just exact matches can be used to return, but now also make it return similar questions.

In [ ]:
# Set the caching to use embeddings
langchain.llm_cache = GPTCache(init_gptcache_embeddings_match)

# Now run it once to warm up the cache
result = llm.invoke("Hello world")


In [ ]:
# Now run a similar request
similar_prompt="Hello world :)"
for i in range(1,10):
    result = llm.invoke(similar_prompt)